In [ ]:
!pip install -q transformers datasets accelerate

import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, f1_score

print("تم تثبيت المكتبات بنجاح! .")

تم تثبيت المكتبات بنجاح! .


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

train_df = pd.read_csv('/content/drive/MyDrive/train.csv')
val_df = pd.read_csv('/content/drive/MyDrive/val.csv')

print(" تم تحميل البيانات بنجاح !")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
 تم تحميل البيانات بنجاح !


In [15]:
train_df = pd.read_csv('/content/drive/MyDrive/train.csv')
val_df = pd.read_csv('/content/drive/MyDrive/val.csv')

train_df = train_df.dropna(subset=['cleaned_text', 'label'])
val_df = val_df.dropna(subset=['cleaned_text', 'label'])

labels = train_df['label'].unique()
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for i, label in enumerate(labels)}

train_df['label_id'] = train_df['label'].map(label2id)
val_df['label_id'] = val_df['label'].map(label2id)

print("التصنيفات ومعرفاتها:", label2id)
print(f"عدد عينات التدريب: {len(train_df)}")
print(f"عدد عينات التحقق: {len(val_df)}")

التصنيفات ومعرفاتها: {'MSA': 0, 'Cairo': 1, 'Doha': 2, 'Tunis': 3, 'Beirut': 4}
عدد عينات التدريب: 46732
عدد عينات التحقق: 5841


In [16]:
model_name = "aubmindlab/bert-base-arabertv02"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(texts):
    return tokenizer(texts, padding="max_length", truncation=True, max_length=128)

print("جاري تحويل النصوص باستخدام الـ Tokenizer...")
train_encodings = tokenizer(list(train_df['cleaned_text']), padding=True, truncation=True, max_length=128)
val_encodings = tokenizer(list(val_df['cleaned_text']), padding=True, truncation=True, max_length=128)

print("تم تحويل النصوص بنجاح وأصبحت جاهزة للتدريب!")

config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/381 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/825k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.64M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

جاري تحويل النصوص باستخدام الـ Tokenizer...
تم تحويل النصوص بنجاح وأصبحت جاهزة للتدريب!


In [17]:
class DialectDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = DialectDataset(train_encodings, list(train_df['label_id']))
val_dataset = DialectDataset(val_encodings, list(val_df['label_id']))

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=5)

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,
    learning_rate=2e-5,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print(" جاري بدء التدريب (Fine-Tuning)...")
trainer.train()

model.safetensors: reconstructing file:   0%|          |  0.00B /  543MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


 جاري بدء التدريب (Fine-Tuning)...


Epoch,Training Loss,Validation Loss,Accuracy
1,0.301405,0.305835,0.890943


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2921, training_loss=0.4325442492084444, metrics={'train_runtime': 654.1123, 'train_samples_per_second': 71.443, 'train_steps_per_second': 4.466, 'total_flos': 1609051721197368.0, 'train_loss': 0.4325442492084444, 'epoch': 1.0})

In [18]:
text_to_test = "إزيك عامل إيه في الشغل؟"

inputs = tokenizer(text_to_test, return_tensors="pt", padding=True, truncation=True, max_length=128)
inputs = {k: v.to(model.device) for k, v in inputs.items()}

model.eval()
with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits
predicted_class_id = logits.argmax(axis=1).item()

predicted_label = id2label[predicted_class_id]

print(f" الجملة: {text_to_test}")
print(f" اللهجة المتوقعة بواسطة AraBERT: {predicted_label}")

 الجملة: إزيك عامل إيه في الشغل؟
 اللهجة المتوقعة بواسطة AraBERT: Cairo


In [19]:
sentences_to_test = [
    "إزيك عامل إيه في الشغل؟",
    "شنو تحب تاكل اليوم؟",
    "كيفك شو اخبارك اليوم؟",
    "شنعمل اليوم؟ العشية خفيفة و الطقس مزيان",
    "تدرس اللغة العربية في الجامعة."
]

print(" جاري اختبار الجمل :\n" + "-"*40)

model.eval()
with torch.no_grad():
    for text in sentences_to_test:

        inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=128)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}


        outputs = model(**inputs)
        logits = outputs.logits
        predicted_class_id = logits.argmax(axis=1).item()


        predicted_label = id2label[predicted_class_id]

        print(f" الجملة: {text}")
        print(f" اللهجة المتوقعة: {predicted_label}\n" + "-"*40)

 جاري اختبار الجمل :
----------------------------------------
 الجملة: إزيك عامل إيه في الشغل؟
 اللهجة المتوقعة: Cairo
----------------------------------------
 الجملة: شنو تحب تاكل اليوم؟
 اللهجة المتوقعة: Doha
----------------------------------------
 الجملة: كيفك شو اخبارك اليوم؟
 اللهجة المتوقعة: Beirut
----------------------------------------
 الجملة: شنعمل اليوم؟ العشية خفيفة و الطقس مزيان
 اللهجة المتوقعة: Tunis
----------------------------------------
 الجملة: تدرس اللغة العربية في الجامعة.
 اللهجة المتوقعة: MSA
----------------------------------------


In [20]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,
    learning_rate=2e-5,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print(" جاري تطوير التدريب (Fine-Tuning - Epoch 3)...")
trainer.train()

 جاري تطوير التدريب (Fine-Tuning - Epoch 3)...


Epoch,Training Loss,Validation Loss,Accuracy
1,0.309082,0.286522,0.894196
2,0.182559,0.320269,0.904297
3,0.128726,0.358737,0.912001


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=8763, training_loss=0.19745626126360868, metrics={'train_runtime': 2064.5295, 'train_samples_per_second': 67.907, 'train_steps_per_second': 4.245, 'total_flos': 4827155163592104.0, 'train_loss': 0.19745626126360868, 'epoch': 3.0})

In [21]:
sentences_to_test = [
    "إزيك عامل إيه في الشغل؟",
    "شنو تحب تاكل اليوم؟",
    "كيفك شو اخبارك اليوم؟",
    "شنعمل اليوم؟ العشية خفيفة و الطقس مزيان",
    "تدرس اللغة العربية في الجامعة.",
    "هو انت رايح فين دلوقتي ومش جاي تاني؟",
    "بربي، وين لقيت الصباط هذا؟ عجبني برشة.",
    "شو بدك نعمل اليوم؟ تعا نحكي شوي بالعشية.",
    "وش سويت في الموضوع اللي قلنا عنه أمس؟",
    "يجب علينا الحفاظ على النظافة العامة في المرافق الحكومية."
]

print(" جاري اختبار الجمل :\n" + "-"*40)

model.eval()
with torch.no_grad():
    for text in sentences_to_test:

        inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=128)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}


        outputs = model(**inputs)
        logits = outputs.logits
        predicted_class_id = logits.argmax(axis=1).item()


        predicted_label = id2label[predicted_class_id]

        print(f" الجملة: {text}")
        print(f" اللهجة المتوقعة: {predicted_label}\n" + "-"*40)

 جاري اختبار الجمل :
----------------------------------------
 الجملة: إزيك عامل إيه في الشغل؟
 اللهجة المتوقعة: Cairo
----------------------------------------
 الجملة: شنو تحب تاكل اليوم؟
 اللهجة المتوقعة: Doha
----------------------------------------
 الجملة: كيفك شو اخبارك اليوم؟
 اللهجة المتوقعة: Beirut
----------------------------------------
 الجملة: شنعمل اليوم؟ العشية خفيفة و الطقس مزيان
 اللهجة المتوقعة: Tunis
----------------------------------------
 الجملة: تدرس اللغة العربية في الجامعة.
 اللهجة المتوقعة: MSA
----------------------------------------
 الجملة: هو انت رايح فين دلوقتي ومش جاي تاني؟
 اللهجة المتوقعة: Cairo
----------------------------------------
 الجملة: بربي، وين لقيت الصباط هذا؟ عجبني برشة.
 اللهجة المتوقعة: Tunis
----------------------------------------
 الجملة: شو بدك نعمل اليوم؟ تعا نحكي شوي بالعشية.
 اللهجة المتوقعة: Beirut
----------------------------------------
 الجملة: وش سويت في الموضوع اللي قلنا عنه أمس؟
 اللهجة المتوقعة: Doha
-----------------------

In [25]:
output_dir = "/content/drive/MyDrive/arabert_dialect_final_model"

model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(" تم حفظ نموذج AraBERT بنجاح على جوجل درايف!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

 تم حفظ نموذج AraBERT بنجاح على جوجل درايف!
